<a href="https://colab.research.google.com/github/ayoosh-bhatta4/teeny-tiny-ships/blob/main/organizer/groq_wrapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
!pip install groq

In [57]:
import os
from groq import Groq
from google.colab import userdata
import json

try:
  groq_key = userdata.get('GROQ_API_KEY')
  os.environ["GROQ_API_KEY"] = groq_key

except userdata.SecretNotFoundError:
  print("Error: ensure that GROQ API KEY is accessible")

client = Groq()


In [58]:
%%writefile test_cases.json
[
    {
        "test_name": "Generalization - Missing Amounts V2",
        "input": "laptop 1200 mouse keyboard 50",
        "expected_sum": 1250
    },
    {
        "test_name": "Generalization - Conversational Story",
        "input": "Paid the plumber 350 to fix the sink and bought a new faucet for 120.",
        "expected_sum": 470
    },
    {
        "test_name": "Generalization - Chaotic Formatting",
        "input": "   sNeAkErS    150   hOoDiE 60  ",
        "expected_sum": 210
    },
    {
        "test_name": "Generalization - Heavy Punctuation",
        "input": "[water_bill]: 45 | (internet_plan) - 60",
        "expected_sum": 105
    },
    {
        "test_name": "Stress Test - Sci-Fi Nonsense",
        "input": "flux_capacitor 5000 kyber_crystal 2000",
        "expected_sum": 7000
    },
    {
        "test_name": "Stress Test - High Volume Hardware",
        "input": "hammer 15 nails 5 drill 80 screws 10 tape 4 paint 30 brushes 12 tarp 8",
        "expected_sum": 164
    },
    {
        "test_name": "Generalization - Long Multi-Word Descriptions",
        "input": "electric bill for january 120 weekend getaway to the mountains 400 fancy dinner date 80",
        "expected_sum": 600
    },
    {
        "test_name": "Edge Case - Number-like Words (Tricky!)",
        "input": "four tickets to the game 200 one dozen donuts 15",
        "expected_sum": 215
    }
]

Overwriting test_cases.json


In [59]:
system_prompt = """
    You are an expert personal finance categorizer.
    Dynamically generate a relevant, standard budgeting category for each item.

    RULES:
    1. Output strictly valid JSON.
    2. The output MUST be a dictionary where the key is the original item name.
    3. The value for each key MUST be an object containing 'category' and 'amount'.
    4. POSITIVE AMOUNTS ONLY: Treat all numbers as positive absolute values. Ignore dashes (-) used as bullet points or separators. Expenses are never negative.
    5. ANTI-HALLUCINATION: If multiple items share a single price at the end (e.g., "mouse keyboard 50"), group them into a single multi-word key. Do NOT duplicate the price across multiple items.

    EXAMPLES OF HOW TO HANDLE EDGE CASES:
    - Input: "metro+auto together 500"
      Logic: This is a single multi-word item.
      Output Key: "metro+auto together" (Amount: 500)

    - Input: "apple 50 banana orange 20"
      Logic: "banana" has no explicit price. "orange" is 20.
      Output Keys: "apple" (Amount: 50), "orange" (Amount: 20). Banana is dropped entirely.

    - Input: "cricket_match - 45 | phone - 60"
      Logic: The dashes are separators, not minus signs. Amounts must be positive.
      Output Keys: "water_bill" (Amount: 45), "internet" (Amount: 60).

    EXPECTED JSON FORMAT:
    {
      "item_name": {"category": "Category Name", "amount": 100}
    }
    """

In [60]:
def organizer_groq(in_str):
  try:
    print("Sending request to Groq...")
    chat_completion = client.chat.completions.create(
        messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": in_str}
        ],
        model="llama-3.3-70b-versatile",
        temperature = 0,
        response_format= {"type": "json_object"}
    )
    raw_json_string = chat_completion.choices[0].message.content

    parsed_items = json.loads(raw_json_string)

    final_totals = {}
    for item, details in parsed_items.items():
      cat = details["category"]
      amt = details["amount"]

      final_totals[cat] = final_totals.get(cat, 0) +amt
    return final_totals

  except Exception as e:
    print(f"Function failed: {e}")
    return{}


In [61]:
import time

def run_evals(test_file_path):
    """Reads the test file and evaluates the model."""
    print(f"--- Starting LLM Evals ---")

    with open(test_file_path, 'r') as file:
        tests = json.load(file)

    passed = 0
    for test in tests:
        print(f"\nRunning Test: {test['test_name']}")
        try:
            # Updated to match your function name: organizer_groq
            result = organizer_groq(test['input'])

            # Calculate the actual sum from the model's output
            actual_sum = sum(result.values())

            # Evaluate
            if actual_sum == test['expected_sum']:
                print("✅ PASS: Math and extraction are perfect.")
                print(f"   Categories created: {list(result.keys())}")
                passed += 1
            else:
                print(f"❌ FAIL: Expected sum {test['expected_sum']}, got {actual_sum}.")
                print(f"   Model Output: {result}")

        except Exception as e:
            print(f"❌ FAIL: Crashed during execution. Error: {e}")

        time.sleep(1)

    print(f"\n--- Evals Complete: {passed}/{len(tests)} Passed ---")

In [62]:
run_evals('test_cases.json')

--- Starting LLM Evals ---

Running Test: Generalization - Missing Amounts V2
Sending request to Groq...
✅ PASS: Math and extraction are perfect.
   Categories created: ['Electronics']

Running Test: Generalization - Conversational Story
Sending request to Groq...
✅ PASS: Math and extraction are perfect.
   Categories created: ['Home Maintenance', 'Home Improvement']

Running Test: Generalization - Chaotic Formatting
Sending request to Groq...
✅ PASS: Math and extraction are perfect.
   Categories created: ['Clothing']

Running Test: Generalization - Heavy Punctuation
Sending request to Groq...
✅ PASS: Math and extraction are perfect.
   Categories created: ['Utilities']

Running Test: Stress Test - Sci-Fi Nonsense
Sending request to Groq...
✅ PASS: Math and extraction are perfect.
   Categories created: ['Electronics', 'Collectibles']

Running Test: Stress Test - High Volume Hardware
Sending request to Groq...
✅ PASS: Math and extraction are perfect.
   Categories created: ['Home Impr